In [141]:
import os.path
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [142]:
patient = pd.read_csv(os.path.join("data", "working_data", "week_5_data.csv")) 
air_pollution = pd.read_csv(os.path.join("data", "week_6", "cri_air_pollution.csv"))

In [143]:
patient.head()

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,CURRENT_AGE,TOTAL_CONDITIONS,TOTAL_ENCOUNTER,ASTHMA_REASON_ENCOUNTER,SYMPTOM_ENCOUNTER,EMERGENCY_ENCOUNTER,URGENT_CARE_ENCOUNTER,ASTHMA_FU_ENCOUNTER,ACTIVE_ALLERGY_COUNT,ACTIVE_MEDICATION_COUNT,ACTIVE_CAREPLAN_COUNT
0,d54a7e5d-02cc-3df8-fa87-a7133bb8dbd0,2019-05-02,NaN,999-87-5214,NaN,NaN,NaN,Ferdinand55,Mitchell808,NaN,NaN,NaN,white,nonhispanic,M,Oxford Massachusetts US,620 Luettgen Avenue,Easthampton,Massachusetts,Hampshire County,25015.0,1027,42.260142,-72.798090,11931.12,0.00,168858,7.0,1,22,6,3,NaN,NaN,3.0,7.0,2,2
1,a8f47b70-4707-d5e3-b3e3-b40c74cfc220,2016-04-05,NaN,999-86-6187,NaN,NaN,NaN,Ariel183,Emmerich580,NaN,NaN,NaN,white,nonhispanic,M,Southborough Massachusetts US,149 Weimann Viaduct Apt 9,Framingham,Massachusetts,Middlesex County,25017.0,1702,42.290083,-71.459564,3255.03,18441.83,12983,10.0,6,33,4,5,2.0,3.0,3.0,8.0,2,3
2,1ebb8037-f48c-5aae-61b8-bca2b28fd86e,2018-04-16,NaN,999-56-5258,NaN,NaN,NaN,Lisa683,Durgan499,NaN,NaN,NaN,white,nonhispanic,F,Quincy Massachusetts US,533 Wuckert Junction Suite 37,Wilmington,Massachusetts,Middlesex County,25017.0,1887,42.573334,-71.150584,8051.91,8174.95,141803,8.0,3,31,1,5,1.0,2.0,NaN,8.0,2,3
3,9ad79ef6-a38d-eec0-14ef-ab987cd94308,2018-01-12,NaN,999-71-4164,NaN,NaN,NaN,Lana840,Maggio310,NaN,NaN,NaN,white,nonhispanic,F,Southbridge Massachusetts US,130 Rodriguez Overpass Apt 96,Lowell,Massachusetts,Middlesex County,25017.0,1850,42.617935,-71.305313,11337.43,316.41,42817,8.0,2,19,1,4,NaN,NaN,NaN,NaN,2,1
4,e6012a16-d3a8-4cbd-a4a2-f94ae5c48685,2016-05-07,NaN,999-53-5147,NaN,NaN,NaN,Chu728,Corwin846,NaN,NaN,NaN,white,nonhispanic,F,Belmont Massachusetts US,559 Greenfelder Manor Apt 45,Holyoke,Massachusetts,Hampden County,25013.0,1040,42.165733,-72.657553,3185.21,28674.85,24016,10.0,7,34,4,5,3.0,2.0,3.0,10.0,2,3


In [144]:
# 1.1	Import the air quality dataset and familiarize yourself with this dataset. Print the number of counties for which the data is available.

display(air_pollution.head())
print(air_pollution["County"].value_counts())

,County,Year,Measurement_avg pm25
0,Abington,2017,10.730195
1,Abington,2018,9.430195
2,Abington,2019,8.730195
3,Abington,2020,8.230195
4,Abington,2021,8.730195


County
Abington       8
Acton          8
Acushnet       8
Adams          8
Agawam         8
              ..
Woburn         8
Worcester      8
Worthington    8
Wrentham       8
Yarmouth       8
Name: count, Length: 351, dtype: int64


In [145]:
# 1.2	Transform the longitudinal air quality dataset into a format that can be merged with cross-sectional tables. 
# Merge it with your data from last week’s exercise (see task 3.1).


air_pollution_wide = air_pollution.pivot_table(
    index="County",
    columns="Year",
    values="Measurement_avg pm25",
    aggfunc="mean"
).reset_index()

# Flatten column names: int years → labelled strings
air_pollution_wide.columns = (
    ["County"] + [f"pm25_{int(y)}" for y in air_pollution_wide.columns[1:]]
)
air_pollution_wide.rename(columns={"County": "COUNTY"}, inplace=True)

In [146]:
air_pollution_wide.head()

,COUNTY,pm25_2017,pm25_2018,pm25_2019,pm25_2020,pm25_2021,pm25_2022,pm25_2023,pm25_2024
0,Abington,10.730195,9.430195,8.730195,8.230195,8.730195,7.930195,7.730195,6.530195
1,Acton,11.000000,10.000000,9.000000,8.500000,9.100000,8.600000,8.400000,7.800000
2,Acushnet,8.431250,7.831250,6.931250,6.531250,7.131250,6.231250,6.231250,5.331250
3,Adams,10.021875,9.821875,8.921875,8.421875,8.821875,8.321875,8.321875,8.621875
4,Agawam,11.367045,10.567045,9.267045,8.667045,10.067045,8.767045,8.767045,8.267045


In [149]:
merged = pd.merge(patient, air_pollution_wide, how="left", left_on="CITY",              
right_on="COUNTY")

# Drop the duplicate and rename the keeper
merged = merged.drop(columns=["COUNTY_x", "COUNTY_y"])

merged.head()   

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,CURRENT_AGE,TOTAL_CONDITIONS,TOTAL_ENCOUNTER,ASTHMA_REASON_ENCOUNTER,SYMPTOM_ENCOUNTER,EMERGENCY_ENCOUNTER,URGENT_CARE_ENCOUNTER,ASTHMA_FU_ENCOUNTER,ACTIVE_ALLERGY_COUNT,ACTIVE_MEDICATION_COUNT,ACTIVE_CAREPLAN_COUNT,pm25_2017,pm25_2018,pm25_2019,pm25_2020,pm25_2021,pm25_2022,pm25_2023,pm25_2024
0,d54a7e5d-02cc-3df8-fa87-a7133bb8dbd0,2019-05-02,NaN,999-87-5214,NaN,NaN,NaN,Ferdinand55,Mitchell808,NaN,NaN,NaN,white,nonhispanic,M,Oxford Massachusetts US,620 Luettgen Avenue,Easthampton,Massachusetts,25015.0,1027,42.260142,-72.798090,11931.12,0.00,168858,7.0,1,22,6,3,NaN,NaN,3.0,7.0,2,2,9.231250,8.431250,7.331250,6.831250,7.931250,7.031250,6.931250,6.531250
1,a8f47b70-4707-d5e3-b3e3-b40c74cfc220,2016-04-05,NaN,999-86-6187,NaN,NaN,NaN,Ariel183,Emmerich580,NaN,NaN,NaN,white,nonhispanic,M,Southborough Massachusetts US,149 Weimann Viaduct Apt 9,Framingham,Massachusetts,25017.0,1702,42.290083,-71.459564,3255.03,18441.83,12983,10.0,6,33,4,5,2.0,3.0,3.0,8.0,2,3,11.332123,10.332123,9.432123,8.832123,9.632123,8.932123,8.732123,7.832123
2,1ebb8037-f48c-5aae-61b8-bca2b28fd86e,2018-04-16,NaN,999-56-5258,NaN,NaN,NaN,Lisa683,Durgan499,NaN,NaN,NaN,white,nonhispanic,F,Quincy Massachusetts US,533 Wuckert Junction Suite 37,Wilmington,Massachusetts,25017.0,1887,42.573334,-71.150584,8051.91,8174.95,141803,8.0,3,31,1,5,1.0,2.0,NaN,8.0,2,3,10.514773,9.314773,8.414773,8.014773,8.714773,8.014773,7.614773,6.814773
3,9ad79ef6-a38d-eec0-14ef-ab987cd94308,2018-01-12,NaN,999-71-4164,NaN,NaN,NaN,Lana840,Maggio310,NaN,NaN,NaN,white,nonhispanic,F,Southbridge Massachusetts US,130 Rodriguez Overpass Apt 96,Lowell,Massachusetts,25017.0,1850,42.617935,-71.305313,11337.43,316.41,42817,8.0,2,19,1,4,NaN,NaN,NaN,NaN,2,1,10.687625,9.587625,8.587625,8.187625,8.887625,8.287625,7.987625,7.387625
4,e6012a16-d3a8-4cbd-a4a2-f94ae5c48685,2016-05-07,NaN,999-53-5147,NaN,NaN,NaN,Chu728,Corwin846,NaN,NaN,NaN,white,nonhispanic,F,Belmont Massachusetts US,559 Greenfelder Manor Apt 45,Holyoke,Massachusetts,25013.0,1040,42.165733,-72.657553,3185.21,28674.85,24016,10.0,7,34,4,5,3.0,2.0,3.0,10.0,2,3,11.128409,10.328409,9.128409,8.628409,9.928409,8.728409,8.728409,8.228409


In [148]:
# # 2.1	Explore the air quality data by using descriptive statistics and data visualization. What relations do you see with your asthma severity measures?
# pm25_cols = [c for c in patient.columns if c.startswith("pm25_")]
# patient["pm25_mean"] = patient[pm25_cols].mean(axis=1)

# severity_cols = [
#     "pm25_mean",
#     "EMERGENCY_ENCOUNTER",
#     "ASTHMA_REASON_ENCOUNTER",
#     "SYMPTOM_ENCOUNTER",
#     "ACTIVE_ALLERGY_COUNT",
#     "INCOME"  # confounder worth seeing early
# ]

# sns.pairplot(
#     df[severity_cols].dropna(),
#     diag_kind="kde",
#     plot_kws={"alpha": 0.4, "s": 15}
# )